# Laboratorium 1 - analiza koszykowa

## Przygotowanie

 * pobierz i wypakuj dataset: https://kaggle.com/datasets/rashikrahmanpritom/groceries-dataset-for-market-basket-analysismba?resource=download&select=basket.csv
   * alternatywnie, pobierz plik `basket.csv` z Teamsów
 * [opcjonalnie] Utwórz wirtualne środowisko
 `python3 -m venv ./recsyslab1`
 * zainstaluj potrzebne biblioteki:
 `pip install more-itertools`

## Część 1. - przygotowanie danych

In [48]:
# importujemy wszystkie potrzebne pakiety

import random
from typing import NamedTuple
from more_itertools import powerset

In [49]:
# definiujemy stałe

PATH = 'data/basket.csv'
EPSILON = 0.001
N = 20          # liczba rekomendacji zwracanych przez recommend()
K = [3, 5, 10]  # wartości k używane przy ewaluacji
N_EVAL = 200    # liczba koszyków użytych do ewaluacji
SEED = 42

In [50]:
# wczytujemy dane o koszykach

def read_baskets(path: str) -> list[set[str]]:
    with open(path) as f:
        raw = f.read()
    baskets = [
        {item.lower() for item in row.split(',') if item}
        for row in raw.split('\n')[1:]
        if row
    ]
    return baskets


def unique_products(baskets: list[set[str]]) -> list[str]:
    products: set[str] = set()
    for basket in baskets:
        products.update(basket)
    return sorted(products)


baskets = read_baskets(PATH)
products = unique_products(baskets)
print(f"Wczytano {len(baskets)} koszyków i {len(products)} unikalnych produktów.")

Wczytano 14963 koszyków i 167 unikalnych produktów.


## Część 2. - obliczanie wskaźników

In [51]:
# obliczamy strukturę danych (słownik) przechowującą wszystkie interesujące wartości `support`

Supports = dict[frozenset, float]


def get_supports(
    baskets: list[set[str]],
    all_products: list[str],
    epsilon: float,
) -> Supports:

    supports = {}

    def get_supports_rec(current_list: list[str], current_idx: int) -> None:
        current_support = len([bsk for bsk in baskets if set(current_list).issubset(bsk)]) / len(baskets)
        if current_support < epsilon:
            return
        supports[frozenset(current_list)] = current_support
        print(f"Set {current_list}: {current_support}")
        for i, product in enumerate(all_products[current_idx+1:]):
            new_list = current_list[:]
            new_list.append(product)
            get_supports_rec(new_list, i + current_idx + 1)

    for i, product in enumerate(products):
        get_supports_rec([product], i)

    return supports


supports = get_supports(baskets, products, EPSILON)
supports

Set ['abrasive cleaner']: 0.0014702933903628951
Set ['artif. sweetener']: 0.0019381140145692708
Set ['baking powder']: 0.008086613646995923
Set ['bathroom cleaner']: 0.0011361358016440553
Set ['beef']: 0.03395041101383412
Set ['beef', 'bottled beer']: 0.0010693042839002875
Set ['beef', 'bottled water']: 0.0013366303548753592
Set ['beef', 'brown bread']: 0.0015371249081066632
Set ['beef', 'butter']: 0.0011361358016440553
Set ['beef', 'canned beer']: 0.0010024727661565194
Set ['beef', 'citrus fruit']: 0.001804450979081735
Set ['beef', 'curd']: 0.0012697988371315912
Set ['beef', 'domestic eggs']: 0.0011361358016440553
Set ['beef', 'frankfurter']: 0.0010024727661565194
Set ['beef', 'frozen vegetables']: 0.0012697988371315912
Set ['beef', 'fruit/vegetable juice']: 0.0010693042839002875
Set ['beef', 'margarine']: 0.001403461872619127
Set ['beef', 'newspapers']: 0.001670787943594199
Set ['beef', 'other vegetables']: 0.002806923745238254
Set ['beef', 'pastry']: 0.0012029673193878234
Set ['beef

{frozenset({'abrasive cleaner'}): 0.0014702933903628951,
 frozenset({'artif. sweetener'}): 0.0019381140145692708,
 frozenset({'baking powder'}): 0.008086613646995923,
 frozenset({'bathroom cleaner'}): 0.0011361358016440553,
 frozenset({'beef'}): 0.03395041101383412,
 frozenset({'beef', 'bottled beer'}): 0.0010693042839002875,
 frozenset({'beef', 'bottled water'}): 0.0013366303548753592,
 frozenset({'beef', 'brown bread'}): 0.0015371249081066632,
 frozenset({'beef', 'butter'}): 0.0011361358016440553,
 frozenset({'beef', 'canned beer'}): 0.0010024727661565194,
 frozenset({'beef', 'citrus fruit'}): 0.001804450979081735,
 frozenset({'beef', 'curd'}): 0.0012697988371315912,
 frozenset({'beef', 'domestic eggs'}): 0.0011361358016440553,
 frozenset({'beef', 'frankfurter'}): 0.0010024727661565194,
 frozenset({'beef', 'frozen vegetables'}): 0.0012697988371315912,
 frozenset({'beef', 'fruit/vegetable juice'}): 0.0010693042839002875,
 frozenset({'beef', 'margarine'}): 0.001403461872619127,
 frozen

In [52]:
# definiujemy funkcje obliczające support, confidence i lift

def support(supports: Supports, products: set[str]) -> float:
    products_frozen = frozenset(products)
    return supports[products_frozen]


def confidence(
    supports: Supports,
    prior: set[str],
    following: set[str],
) -> float:
    return supports[frozenset(set().union(prior).union(following))] / supports[frozenset(prior)]


def lift(
    supports: Supports,
    prior: set[str],
    following: set[str],
) -> float:
    return supports[frozenset(set().union(prior).union(following))] / (supports[frozenset(prior)] * supports[frozenset(following)])

In [53]:
print(support(supports, {'whole milk', 'rolls/buns'}))
print(confidence(supports, {'whole milk', 'rolls/buns'}, {'yogurt'}))
print(lift(supports, {'whole milk', 'rolls/buns'}, {'yogurt'}))

0.013967787208447505
0.09569377990430622
1.1142926293448512


## Część 3. - generowanie rekomendacji

In [54]:
# wyznaczamy listę potencjalnych rekomendacji
# rekomendowane artykuły powinny mieć lift > 1 i jak najwyższe confidence

class Candidate(NamedTuple):
    item: str
    subbasket: set[str]
    confidence: float
    lift: float


def generate_next_product_candidates(
    basket: set[str],
    products: list[str],
    supports: Supports,
) -> list[Candidate]:
    """Zwraca listę kandydatów posortowanych malejąco po confidence."""
    candidates = []
    for sub_basket in powerset(basket):
        if sub_basket == tuple():
            continue
        sub_basket_set = set(sub_basket)
        for product in products:
            if product in basket:
                continue
            new_basket = sub_basket_set | {product}
            if frozenset(new_basket) in supports and (curr_lift := lift(supports, sub_basket_set, {product})) > 1:
                curr_confidence = confidence(supports, sub_basket_set, {product})
                curr_candidate = Candidate(product, sub_basket_set, curr_confidence, curr_lift)
                candidates.append(curr_candidate)

    candidates.sort(key=lambda x: x.confidence, reverse=True)
    return candidates

In [55]:
# wybieramy top-n unikalnych produktów z listy kandydatów

def recommend(
    basket: set[str],
    products: list[str],
    supports: Supports,
    n: int,
) -> list[str]:
    """Zwraca top-n rekomendacji na podstawie analizy koszykowej."""
    recommendations = []
    recommendations_set = set()

    candidates = generate_next_product_candidates(basket, products, supports)
    for i, candidate in enumerate(candidates):
        if len(recommendations) == n:
            break
        if candidate.item not in recommendations_set:
            recommendations.append(candidate.item)
            recommendations_set.add(candidate.item)

    return recommendations

In [56]:
print("Koszyk:", baskets[1])
print("Rekomendacje:", recommend(baskets[1], products, supports, N))

Koszyk: {'whole milk', 'semi-finished bread', 'yogurt', 'sausage'}
Rekomendacje: ['rolls/buns', 'soda', 'bottled beer', 'citrus fruit', 'pastry', 'curd', 'frozen vegetables', 'beverages', 'dessert', 'frozen meals', 'salty snack', 'sliced cheese', 'grapes', 'misc. beverages', 'ham', 'chewing gum', 'hard cheese', 'soft cheese', 'specialty bar', 'herbs']


In [57]:
print("Koszyk:", baskets[33])
print("Rekomendacje:", recommend(baskets[33], products, supports, N))

Koszyk: {'white wine', 'yogurt', 'tropical fruit', 'domestic eggs', 'photo/film', 'root vegetables', 'soda'}
Rekomendacje: ['sausage', 'citrus fruit', 'shopping bags', 'newspapers', 'coffee', 'frankfurter', 'frozen vegetables', 'white bread', 'uht-milk', 'specialty chocolate', 'beverages', 'meat', 'waffles', 'oil', 'sliced cheese', 'chewing gum', 'flour', 'grapes', 'processed cheese', 'cat food']


## Część 4. - ewaluacja rekomendacji: MRR@k

### Metryka

**MRR@k** (Mean Reciprocal Rank) mierzy, na jakiej *pozycji* pojawił się ukryty produkt wśród top-k rekomendacji. Nagradza wyższe miejsca na liście:

$$\text{MRR@k} = \frac{1}{|\mathcal{Q}|} \sum_{q \in \mathcal{Q}} \frac{1}{\text{rank}_q}$$

gdzie:
- $\mathcal{Q}$ to zbiór przypadków testowych
- $\text{rank}_q$ to pozycja pierwszego trafnego produktu w top-k dla danego przypadku testowego $q$ (lub $\infty$, jeśli nie znaleziono — wkład do sumy wynosi wtedy 0).

### Protokół ewaluacji

Dla każdego koszyka testowego:
1. losowo wydzielamy jeden ukryty produkt; ten produkt byłby tym, który użytkownik najchętniej umieści w koszyku i tym samym tym, który powinien zaproponować nasz algorytm,
2. pozostałe produkty stanowią kontekst (czyli koszyk) przekazywany rekomendatorowi,
3. sprawdzamy, na której pozycji top-k pojawia się ukryty produkt.

Porównamy dwa algorytmy:
- **basket** – rekomendator oparty na analizie koszykowej (confidence + lift),
- **random** – rekomendator losowy (baseline).

Porównanie możemy przeprowadzić dla różnych wartości $k$.

In [58]:
# algorytm losowy — punkt odniesienia (baseline)

def recommend_random(
    basket: set[str],
    products: list[str],
    k: int,
    rng: random.Random | None = None,
) -> list[str]:
    """Zwraca k losowo wybranych produktów spoza koszyka."""
    if rng is None:
        rng = random.Random()
    return rng.sample([p for p in products if p not in basket], k)

In [59]:
# implementacja metryki MRR@k

def reciprocal_rank_at_k(recommended: list[str], relevant: set[str], k: int) -> float:
    """Zwraca odwrotność pozycji pierwszego trafnego produktu w top-k (0 jeśli brak trafienia)."""
    for i, item in enumerate(recommended[:k], start=1):
        if item in relevant:
            return 1.0 / i
    return 0.0

In [60]:
# podział koszyka na kontekst i ukryte produkty (ground truth)

def split_basket(
    basket: set[str],
    sample_size: int,
    rng: random.Random,
) -> tuple[set[str], set[str]]:
    """Losowo wydziela `sample_size` produktów jako cel dla rekomendacji; reszta to koszyk"""
    relevant = set(rng.sample(list(basket), sample_size))
    context = basket - relevant
    return context, relevant

In [61]:
# typy reprezentujące wyniki ewaluacji

class MetricAtK(NamedTuple):
    k: int
    mrr: float


class EvaluationResults(NamedTuple):
    basket: MetricAtK
    random: MetricAtK

In [62]:
# ewaluacja obu rekomendatorów dla zadanego k

def evaluate(
    baskets: list[set[str]],
    products: list[str],
    supports: Supports,
    k: int,
    n_eval: int = N_EVAL,
    seed: int = SEED,
) -> EvaluationResults:
    """
    Generuje `n_eval` przypadków testowych przy uzyciu funkcji `split_basket`.
    Dla kazdego przypadku testowego uruchamia oba algorytmy, a następnie oblicza `MMR@k`.
    Zwraca średnie wartości MMR@k dla wskazanego `k` i dla obu algorytmów.
    """
    mrr_basket = 0.0
    mrr_random = 0.0

    baskets = [basket for basket in baskets if len(basket) > 1]

    rng = random.Random(seed)

    test_cases = [split_basket(rng.choice(baskets), 1, rng) for _ in range(n_eval)]

    for context, relevant in test_cases:
        recommended_random = recommend_random(context, products, k, rng)
        recommended_basket = recommend(context, products, supports, k)

        mrr_basket += reciprocal_rank_at_k(recommended_basket, relevant, k)
        mrr_random += reciprocal_rank_at_k(recommended_random, relevant, k)
        
    return EvaluationResults(MetricAtK(k, mrr_basket / n_eval), MetricAtK(k, mrr_random / n_eval))


In [63]:
# wyświetlamy wyniki i porównujemy algorytmy dla każdego k z listy K

def print_evaluation(results: EvaluationResults) -> None:
    col_w = 10
    header = f"{'Algorytm':<12} | {'mrr@' + str(results.basket.k):>{col_w}}"
    print(header)
    print('-' * len(header))
    for algo, metric in results._asdict().items():
        print(f"{algo:<12} | {metric.mrr:>{col_w}.4f}")


for k in K:
    print_evaluation(evaluate(baskets, products, supports, k))
    print()

Algorytm     |      mrr@3
-------------------------
basket       |     0.0450
random       |     0.0067

Algorytm     |      mrr@5
-------------------------
basket       |     0.0483
random       |     0.0162

Algorytm     |     mrr@10
-------------------------
basket       |     0.0498
random       |     0.0144

